In [35]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
import os

load_dotenv(
    r"C:\Users\shekh\OneDrive\Desktop\Tutorial Agentic AI\.env",
    override=True
)

assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found in .env"

print("Imports and API key loaded successfully!")

Imports and API key loaded successfully!


In [36]:
PRODUCTS = {
    "wireless headphones": {
        "price": 79.99,
        "rating": 4.6,
        "stock": 25,
        "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."
    },
    "smart watch": {
        "price": 199.99,
        "rating": 4.3,
        "stock": 12,
        "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."
    },
    "mechanical keyboard": {
        "price": 129.00,
        "rating": 4.8,
        "stock": 40,
        "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."
    },
    "laptop stand": {
        "price": 34.99,
        "rating": 4.5,
        "stock": 18,
        "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."
    },
}

print("Product database loaded!")

Product database loaded!


In [37]:
@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""

    product = PRODUCTS.get(name.strip().lower())

    if not product:
        return (
            f"Product not found. Available products: "
            f"{', '.join(PRODUCTS.keys())}"
        )

    return f"Product: {name}\nDetails: {product}"

In [38]:
REVIEWS = {
    "wireless headphones": {
        "reviews": 1262,
        "rating": 4.6
    },
    "smart watch": {
        "reviews": 340,
        "rating": 3.9
    },
    "mechanical keyboard": {
        "reviews": 67,
        "rating": 4.8
    },
    "laptop stand": {
        "reviews": 781,
        "rating": 4.5
    },
}

print("Review database loaded!")

Review database loaded!


In [39]:
@tool
def get_review(name: str) -> str:
    """Look up customer reviews by product name and return the review count and customer rating."""

    review = REVIEWS.get(name.strip().lower())

    if not review:
        return f"Reviews are not available for {name}."

    return f"Product: {name}\nReview details: {review}"

In [40]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_retries=0
)

print("Groq model initialized successfully!")

Groq model initialized successfully!


In [41]:
memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[get_product, get_review],
    system_prompt="""
    You are a helpful product assistant for an online technology store.

    Instructions:
    1. Use get_product for product prices, stock, ratings and descriptions.
    2. Use get_review for customer review counts and review ratings.
    3. Use both tools when a question asks for both product details and reviews.
    4. Never invent product information.
    5. Clearly distinguish product ratings from customer review ratings.
    6. If a product is not found, explain which products are available.
    7. Answer clearly and concisely.
    """,
    checkpointer=memory,
)

print("Product query agent created successfully!")

Product query agent created successfully!


In [42]:
config = {
    "configurable": {
        "thread_id": "product-chat-1"
    }
}

def ask(question: str):
    result = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": question}
            ]
        },
        config=config
    )

    print(result["messages"][-1].content)

In [43]:
ask("What is the price of wireless headphones?")

The wireless headphones are priced at **$79.99**.


In [44]:
ask("Tell me about the mechanical keyboard.")

Here are the details for the **Mechanical Keyboard**:

- **Price:** $129.00  
- **Rating:** 4.8 / 5 (product rating)  
- **Stock:** 40 units available  
- **Description:** Tenkeyless layout with Cherry MX Brown switches and per‑key RGB lighting.


In [45]:
ask("How do people like the smartwatch?")

Customer review data for the **Smartwatch** isn’t available at the moment, so I can’t provide information on how people have rated it. If you’d like details such as price, stock, or product specifications, just let me know!


In [46]:
ask("What is the price and review information for the smartwatch?")

**Smart Watch**

- **Price:** $199.99  
- **Product Rating:** 4.3 / 5 (manufacturer’s rating)  
- **Customer Reviews:** 340 reviews with an average **customer rating of 3.9 / 5**.  

Let me know if you’d like any additional details!


In [47]:
ask("Tell me the price of smartwatch?")

The smartwatch is priced at **$199.99**.


In [48]:
ask("What was it review?")

The smartwatch has **340 customer reviews** with an average **customer rating of 3.9 / 5**.
